# BM25: a hands-on implementation

**Track:** Search · **Guide:** [`044-bm25.md`](../guides/concepts/044-bm25.md)

BM25 is a probabilistic lexical ranking function that accounts for term frequency, rarity, and document length.

> **Intuition:** A long document should not win merely because it repeats a query word many times.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement BM25]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Retrieve exact terminology and identifiers in manuals where vector similarity can miss a critical literal match.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Tokenize the corpus and calculate statistics

BM25 combines query-term frequency, inverse document frequency, and document-length normalization.


In [ ]:
import math
import re
from collections import Counter

documents = {
    "a": "reset password from account settings",
    "b": "account billing address update",
    "c": "reset password reset account password",
}
tokens = {key: re.findall(r"[a-z]+", text.lower()) for key, text in documents.items()}
average_length = sum(map(len, tokens.values())) / len(tokens)
document_frequency = Counter(
    term for words in tokens.values() for term in set(words)
)
print("average document length:", average_length)

## 2. Implement BM25 scoring

The usual IDF variant below stays positive for rare terms; `k1` controls term-frequency saturation and `b` controls length normalization.


In [ ]:
def bm25(query, document_id, k1=1.5, b=0.75):
    query_terms = set(re.findall(r"[a-z]+", query.lower()))
    words = tokens[document_id]
    counts = Counter(words)
    score = 0.0
    for term in query_terms:
        frequency = counts[term]
        if not frequency:
            continue
        df = document_frequency[term]
        idf = math.log(1 + (len(documents) - df + 0.5) / (df + 0.5))
        norm = frequency + k1 * (1 - b + b * len(words) / average_length)
        score += idf * frequency * (k1 + 1) / norm
    return score

## 3. Rank and inspect the results

BM25 is lexical: exact terms and rare identifiers matter. Evaluate ranking quality on labeled queries rather than assuming a formula guarantees relevance.


In [ ]:
query = "reset password"
ranking = sorted(
    ((bm25(query, document_id), document_id) for document_id in documents),
    reverse=True,
)
for score, document_id in ranking:
    print(f"{score:.3f} {document_id}: {documents[document_id]}")

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/044-bm25.md) and the
[learning roadmap](../guides/ROADMAP.md).
